In [ ]:
import pandas as pd

df = pd.read_csv('/content/kamayani_corpus.csv')

print("Total passages:", len(df))
print(df.groupby('sarg').size())

display(df.head())

In [ ]:
import re

chunks = []

for _, row in df.iterrows():
    passage_text = str(row['text'])

    # Split text wherever blank lines occur
    parts = re.split(r'\n\s*\n', passage_text)

    for i, part in enumerate(parts):
        part = part.strip()

        if len(part) > 40:
            chunks.append({
                'chunk_id': f"{row['passage_id']}_chunk_{i+1}",
                'passage_id': row['passage_id'],
                'sarg': row['sarg'],
                'text': part,
                'source_url': row['source_url']
            })

chunks_df = pd.DataFrame(chunks)

print("Original passages:", len(df))
print("Total chunks:", len(chunks_df))

display(chunks_df.head())

In [ ]:
print("Chunks per Sarg:")
print(chunks_df.groupby('sarg').size())

chunks_df['text_length'] = chunks_df['text'].str.len()

print("\nChunk length statistics:")
print(chunks_df['text_length'].describe())

display(chunks_df[['chunk_id', 'sarg', 'text_length']])

In [ ]:
refined_chunks = []

max_chars = 500

for _, row in chunks_df.iterrows():
    lines = [line.strip() for line in row['text'].splitlines() if line.strip()]

    current_lines = []
    current_length = 0
    chunk_num = 1

    for line in lines:
        if current_lines and current_length + len(line) + 1 > max_chars:
            refined_chunks.append({
                'chunk_id': f"{row['passage_id']}_refined_{chunk_num}",
                'passage_id': row['passage_id'],
                'sarg': row['sarg'],
                'text': '\n'.join(current_lines),
                'source_url': row['source_url']
            })
            chunk_num += 1
            current_lines = []
            current_length = 0

        current_lines.append(line)
        current_length += len(line) + 1

    if current_lines:
        refined_chunks.append({
            'chunk_id': f"{row['passage_id']}_refined_{chunk_num}",
            'passage_id': row['passage_id'],
            'sarg': row['sarg'],
            'text': '\n'.join(current_lines),
            'source_url': row['source_url']
        })

refined_df = pd.DataFrame(refined_chunks)

print("Refined chunks:", len(refined_df))
print("Maximum length:", refined_df['text'].str.len().max())
display(refined_df[['chunk_id', 'sarg', 'text']].head())

In [ ]:
# Assign globally unique IDs to all refined chunks
refined_df = refined_df.reset_index(drop=True)

refined_df['chunk_id'] = [
    f"KMY_{i:03d}" for i in range(1, len(refined_df) + 1)
]

print("Total chunks:", len(refined_df))
print("Unique IDs:", refined_df['chunk_id'].nunique())
print("Duplicate IDs:", refined_df['chunk_id'].duplicated().sum())

display(refined_df[['chunk_id', 'passage_id', 'sarg']].head(10))

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2'
)

print("Embedding model loaded successfully!")

In [ ]:
texts = refined_df['text'].tolist()

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings created successfully!")
print("Embedding shape:", embeddings.shape)

In [ ]:
import numpy as np

def retrieve_passages(query, top_k=3):

    # Convert question into an embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Calculate similarity with all poetry chunks
    similarities = np.dot(embeddings, query_embedding)

    # Find top matching chunks
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = refined_df.iloc[top_indices].copy()
    results['similarity'] = similarities[top_indices]

    return results[['chunk_id', 'sarg', 'similarity', 'text', 'source_url']]

In [ ]:
query = "कामायनी में नदी, जल और पानी का वर्णन कहाँ मिलता है?"

results = retrieve_passages(query, top_k=3)

display(results)

In [ ]:
test_queries = [
    "कामायनी में जल और नदियों का वर्णन",
    "कविता में पर्वत और हिमालय का चित्रण",
    "प्रकृति और मनुष्य के बीच संबंध",
    "वनस्पतियों और पेड़ पौधों का वर्णन",
    "प्रलय और प्रकृति के विनाश का चित्रण"
]

for q in test_queries:
    print("\nQUERY:", q)
    display(retrieve_passages(q, top_k=3))

In [ ]:
for q in test_queries:

    print("\n" + "="*70)
    print("QUERY:", q)
    print("="*70)

    results = retrieve_passages(q, top_k=3)

    for rank, (_, row) in enumerate(results.iterrows(), start=1):

        print(f"\nRank: {rank}")
        print("Chunk ID:", row['chunk_id'])
        print("Sarg:", row['sarg'])
        print("Similarity:", round(row['similarity'], 3))
        print("Full Text:\n", row['text'])

In [ ]:
evaluation = pd.DataFrame([
    ["Water and rivers", "KMY_007", "Relevant"],
    ["Water and rivers", "KMY_013", "Relevant"],
    ["Mountains and Himalaya", "KMY_021", "Highly Relevant"],
    ["Human-nature relationship", "KMY_025", "Relevant"],
    ["Human-nature relationship", "KMY_007", "Relevant"],
    ["Human-nature relationship", "KMY_043", "Relevant"],
    ["Vegetation", "KMY_003", "Partially Relevant"],
    ["Vegetation", "KMY_025", "Relevant"],
    ["Vegetation", "KMY_043", "Partially Relevant"],
    ["Destruction and Pralaya", "KMY_007", "Highly Relevant"],
    ["Destruction and Pralaya", "KMY_001", "Highly Relevant"],
    ["Destruction and Pralaya", "KMY_010", "Highly Relevant"]
], columns=["query", "chunk_id", "manual_relevance"])

display(evaluation)

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')

print("API key loaded:", bool(api_key))

In [ ]:
from google import genai

client = genai.Client(api_key=api_key)

for m in client.models.list():
    print(m.name)

In [ ]:
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY').strip()

print("API key loaded:", bool(api_key))

In [ ]:
from google import genai

client = genai.Client(api_key=api_key)

print("Gemini client created successfully!")

In [ ]:
for m in client.models.list():
    print(m.name)

In [ ]:
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY').strip()

from google import genai

client = genai.Client(api_key=api_key)

print("Gemini client created successfully!")

In [ ]:
for m in client.models.list():
    print(m.name)

In [ ]:
from google import genai

client = genai.Client(api_key=api_key)

def generate_rag_answer(query, top_k=3):

    # Step 1: Retrieve relevant poetry passages
    retrieved = retrieve_passages(query, top_k=top_k)

    # Step 2: Prepare context
    context = "\n\n".join(
        f"[{row.chunk_id} | {row.sarg}]\n{row.text}"
        for _, row in retrieved.iterrows()
    )

    # Step 3: Create prompt
    prompt = f"""
तुम हिंदी साहित्य के शोध सहायक हो।

नीचे दिए गए कामायनी के अंशों के आधार पर ही उत्तर दो।

नियम:
1. केवल दिए गए अंशों से उत्तर तैयार करो।
2. कविता की पंक्तियाँ या तथ्य मत गढ़ो।
3. यदि पर्याप्त जानकारी नहीं है तो स्पष्ट बताओ।
4. उत्तर सरल हिंदी में दो।
5. उत्तर में संबंधित Chunk IDs का उल्लेख करो।

प्रश्न: {query}

कविता के प्राप्त अंश:
{context}
"""

    # Step 4: Generate answer
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text, retrieved

In [ ]:
def generate_rag_answer(query, top_k=3):

    retrieved = retrieve_passages(query, top_k=top_k)

    context = "\n\n".join(
        f"[{row.chunk_id} | {row.sarg}]\n{row.text}"
        for _, row in retrieved.iterrows()
    )

    prompt = f"""
तुम हिंदी साहित्य के शोध सहायक हो।

नीचे दिए गए कामायनी के अंशों के आधार पर ही उत्तर दो।

नियम:
1. केवल दिए गए अंशों से उत्तर दो।
2. कोई कविता या तथ्य मत गढ़ो।
3. जानकारी पर्याप्त न हो तो स्पष्ट बताओ।
4. सरल हिंदी में उत्तर दो।
5. इस्तेमाल किए गए Chunk IDs बताओ।

प्रश्न: {query}

प्राप्त कविता के अंश:
{context}
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    return response.text, retrieved

In [ ]:
answer, sources = generate_rag_answer(
    "कामायनी में हिमालय और प्रकृति का वर्णन किस प्रकार किया गया है?"
)

print("GENERATED ANSWER:\n")
print(answer)

print("\nRETRIEVED SOURCES:\n")
display(sources[['chunk_id', 'sarg', 'similarity']])

In [ ]:
import pandas as pd

flora = [
    "वृक्ष", "पेड़", "वन", "लता", "पुष्प",
    "फूल", "पल्लव", "पत्ते", "वनस्पति",
    "तृण", "कमल", "धान", "घास"
]

fauna = [
    "पक्षी", "खग", "मृग", "पशु",
    "हंस", "जीव", "प्राणी", "मछली"
]

def count_terms(text, terms):
    return {term: text.count(term) for term in terms}

all_text = " ".join(refined_df["text"].fillna("").astype(str))

flora_counts = count_terms(all_text, flora)
fauna_counts = count_terms(all_text, fauna)

frequency_df = pd.DataFrame(
    [(term, count, "Flora") for term, count in flora_counts.items()] +
    [(term, count, "Fauna") for term, count in fauna_counts.items()],
    columns=["Term", "Frequency", "Category"]
)

frequency_df = frequency_df[frequency_df["Frequency"] > 0]
frequency_df = frequency_df.sort_values(
    "Frequency", ascending=False
)

display(frequency_df)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.bar(
    frequency_df["Term"],
    frequency_df["Frequency"]
)

plt.title("Flora and Fauna References in Kamayani")
plt.xlabel("Ecological Terms")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
frequency_df.to_csv(
    "flora_fauna_frequency.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Frequency CSV saved successfully!")

In [ ]:
from itertools import combinations
from collections import Counter, defaultdict
import pandas as pd

# Flora and fauna terms
flora = [
    "वृक्ष", "पेड़", "वन", "लता", "पुष्प",
    "फूल", "पल्लव", "पत्ते", "वनस्पति",
    "तृण", "कमल", "धान", "घास"
]

fauna = [
    "पक्षी", "खग", "मृग", "पशु",
    "हंस", "जीव", "प्राणी", "मछली"
]

all_terms = flora + fauna

# Count terms and co-occurrence across chunks
node_counts = Counter()
edge_counts = Counter()

for text in refined_df["text"].fillna("").astype(str):
    present = [term for term in all_terms if term in text]

    for term in present:
        node_counts[term] += text.count(term)

    for pair in combinations(sorted(set(present)), 2):
        edge_counts[pair] += 1

# Create Nodes table
nodes_df = pd.DataFrame([
    {
        "Id": term,
        "Label": term,
        "Category": "Flora" if term in flora else "Fauna",
        "Frequency": count
    }
    for term, count in node_counts.items()
])

# Create Edges table
edges_df = pd.DataFrame([
    {
        "Source": pair[0],
        "Target": pair[1],
        "Weight": weight,
        "Type": "Undirected"
    }
    for pair, weight in edge_counts.items()
])

display(nodes_df)
display(edges_df)

print("Total nodes:", len(nodes_df))
print("Total edges:", len(edges_df))

In [ ]:
nodes_df.to_csv("kamayani_nodes.csv", index=False, encoding="utf-8-sig")
edges_df.to_csv("kamayani_edges.csv", index=False, encoding="utf-8-sig")

from google.colab import files

files.download("kamayani_nodes.csv")
files.download("kamayani_edges.csv")

In [ ]:
display(nodes_df.head())
print(nodes_df.columns)

In [ ]:
nodes_df = frequency_df.copy()

nodes_df = nodes_df.rename(columns={
    "Term": "Label"
})

nodes_df.insert(0, "Id", nodes_df["Label"])

display(nodes_df)

In [ ]:
print("Available variables:")

for name in ["df", "refined_df", "frequency_df", "nodes_df"]:
    print(name, ":", name in globals())